In [1]:
import pandas as pd
import re
import os
import sys
from datetime import datetime, timedelta
from plotnine import *

In [2]:
# This script is intended to add new samples
# Either from public databases (NCBI and GISAID)
# Or from Nicole K. 

In [3]:
# Import various metadatas

metadata = pd.read_excel("./metadata/global-H5N5-A6-seq-n334-metadata.xlsx")
gisaid = pd.read_excel("./database_pulls/2026-03-11/gisaid_epiflu_isolates.xls")
ncbi = pd.read_csv("./database_pulls/2026-03-11/sequences.csv", sep = ",")

In [4]:
print(len(metadata))
print(len(gisaid))
print(len(ncbi))

342
407
344


In [5]:
#Function: Read in the FASTA as a dataframe

def fasta_reader(path_to_fasta, output_name):
    fasta_data = []
    FASTANAME = path_to_fasta
        
    with open(FASTANAME) as f:
        header = ""
        sequence = ""
        for line in f:
            if line.startswith(">"):
                if header != "":
                    fasta_data.append({"header": header, "sequence": sequence})
                header = line.strip() 
                sequence = ""
            else:
                sequence += line.strip()
        fasta_data.append({"header": header, "sequence": sequence}) #last line 
        
    globals()[output_name] = pd.DataFrame(fasta_data)

    return

In [6]:
#Function: write a new FASTA with updated header (function written by Maria)  
def fasta_writer(path, filename, df):
            
    try:  
        os.mkdir(path)

    except OSError as error:
        pass

    with open(f"{path}{filename}", "w") as f:
        for index, row in df.iterrows():
            f.write(f"{row['identifier']}\n")
            f.write(f"{row['sequence']}\n")

In [129]:
# Step One: Determine which of these H5N5 sequences are A6

def filter_genotype(FASTA1, FASTA2): #, gisaid_semgents, ncbi_segments):

    # Read in the FASTAs
    fasta_reader(FASTA1, "gisaid_fasta") 
    fasta_reader(FASTA2, "ncbi_fasta")

    # Make an identifier and segment column
    gisaid_fasta["identifier"] = ">" + gisaid_fasta["header"].str.split("|", expand = True)[2]
    gisaid_fasta["segment"] = gisaid_fasta["header"].str.rsplit("|", n=1).str[-1]

    ncbi_fasta["identifier"] = ">" + ncbi_fasta["header"].str.split("(", expand = True)[1]
    ncbi_fasta["segment"] = ncbi_fasta["header"].str.extract(r"segment\s(\d{1})")

    # Divide FASTAs by segment
    pb2_gisaid = gisaid_fasta[gisaid_fasta["segment"] == "PB2"]
    pb1_gisaid = gisaid_fasta[gisaid_fasta["segment"] == "PB1"]
    pa_gisaid = gisaid_fasta[gisaid_fasta["segment"] == "PA"]
    ha_gisaid = gisaid_fasta[gisaid_fasta["segment"] == "HA"]
    np_gisaid = gisaid_fasta[gisaid_fasta["segment"] == "NP"]
    na_gisaid = gisaid_fasta[gisaid_fasta["segment"] == "NA"]
    mp_gisaid = gisaid_fasta[gisaid_fasta["segment"] == "MP"]
    ns_gisaid = gisaid_fasta[gisaid_fasta["segment"] == "NS"]

    pb2_ncbi = ncbi_fasta.loc[ncbi_fasta["segment"] == "1"].drop_duplicates(subset="identifier", keep="first")
    pb1_ncbi = ncbi_fasta.loc[ncbi_fasta["segment"] == "2"].drop_duplicates(subset="identifier", keep="first")
    pa_ncbi = ncbi_fasta.loc[ncbi_fasta["segment"] == "3"].drop_duplicates(subset="identifier", keep="first")
    ha_ncbi = ncbi_fasta.loc[ncbi_fasta["segment"] == "4"].drop_duplicates(subset="identifier", keep="first") 
    np_ncbi = ncbi_fasta.loc[ncbi_fasta["segment"] == "5"].drop_duplicates(subset="identifier", keep="first")
    na_ncbi = ncbi_fasta.loc[ncbi_fasta["segment"] == "6"].drop_duplicates(subset="identifier", keep="first")
    mp_ncbi = ncbi_fasta.loc[ncbi_fasta["segment"] == "7"].drop_duplicates(subset="identifier", keep="first")
    ns_ncbi = ncbi_fasta.loc[ncbi_fasta["segment"] == "8"].drop_duplicates(subset="identifier", keep="first")

    # Merge each segment
    pb2_concat_h5n5 = pd.concat([pb2_gisaid, pb2_ncbi], join = "outer")
    pb1_concat_h5n5 = pd.concat([pb1_gisaid, pb1_ncbi], join = "outer")
    pa_concat_h5n5 = pd.concat([pa_gisaid, pa_ncbi], join = "outer")
    ha_concat_h5n5 = pd.concat([ha_gisaid, ha_ncbi], join = "outer")
    np_concat_h5n5 = pd.concat([np_gisaid, np_ncbi], join = "outer")
    na_concat_h5n5 = pd.concat([na_gisaid, na_ncbi], join = "outer")
    mp_concat_h5n5 = pd.concat([mp_gisaid, mp_ncbi], join = "outer")
    ns_concat_h5n5 = pd.concat([ns_gisaid, ns_ncbi], join = "outer")

    print(ha_concat_h5n5["identifier"].is_unique)

    fasta_dict = {"pb2": pb2_concat_h5n5, "pb1": pb1_concat_h5n5, "pa": pa_concat_h5n5, "ha": ha_concat_h5n5, "np": np_concat_h5n5, "na": na_concat_h5n5, "mp": mp_concat_h5n5, "ns": ns_concat_h5n5}

    # Clean things up
    pb2_concat_h5n5["sequence"] = pb2_concat_h5n5["sequence"].str.upper()
    pb1_concat_h5n5["sequence"] = pb1_concat_h5n5["sequence"].str.upper()
    pa_concat_h5n5["sequence"] = pa_concat_h5n5["sequence"].str.upper()
    ha_concat_h5n5["sequence"] = ha_concat_h5n5["sequence"].str.upper()
    np_concat_h5n5["sequence"] = np_concat_h5n5["sequence"].str.upper()
    na_concat_h5n5["sequence"] = na_concat_h5n5["sequence"].str.upper()
    mp_concat_h5n5["sequence"] = mp_concat_h5n5["sequence"].str.upper()
    ns_concat_h5n5["sequence"] = ns_concat_h5n5["sequence"].str.upper()

    pb2_concat_h5n5["identifier"] = pb2_concat_h5n5["identifier"].str.replace(" ", "_")
    pb1_concat_h5n5["identifier"] = pb1_concat_h5n5["identifier"].str.replace(" ", "_")
    pa_concat_h5n5["identifier"] = pa_concat_h5n5["identifier"].str.replace(" ", "_")
    ha_concat_h5n5["identifier"] = ha_concat_h5n5["identifier"].str.replace(" ", "_")
    np_concat_h5n5["identifier"] = np_concat_h5n5["identifier"].str.replace(" ", "_")
    na_concat_h5n5["identifier"] = na_concat_h5n5["identifier"].str.replace(" ", "_")
    mp_concat_h5n5["identifier"] = mp_concat_h5n5["identifier"].str.replace(" ", "_")
    ns_concat_h5n5["identifier"] = ns_concat_h5n5["identifier"].str.replace(" ", "_")

    # Print new FASTAs
    fasta_writer("./database_pulls/2026-03-11/", "pb2.fasta", pb2_concat_h5n5)
    fasta_writer("./database_pulls/2026-03-11/", "pb1.fasta", pb1_concat_h5n5)
    fasta_writer("./database_pulls/2026-03-11/", "pa.fasta", pa_concat_h5n5)
    fasta_writer("./database_pulls/2026-03-11/", "ha.fasta", ha_concat_h5n5)
    fasta_writer("./database_pulls/2026-03-11/", "np.fasta", np_concat_h5n5)
    fasta_writer("./database_pulls/2026-03-11/", "na.fasta", na_concat_h5n5)
    fasta_writer("./database_pulls/2026-03-11/", "mp.fasta", mp_concat_h5n5)
    fasta_writer("./database_pulls/2026-03-11/", "ns.fasta", ns_concat_h5n5)

    return(fasta_dict)

In [130]:
concatenated_fastas = filter_genotype("./database_pulls/2026-03-11/gisaid_epiflu_sequence.fasta", "./database_pulls/2026-03-11/sequences.fasta")

True


In [ ]:
# Here, run the per-segment FASTA files you just make through GenoFlu to generate the results

In [213]:
# Step Two: Generate a list of strain names that are not in the current metadata

def identify_new_public_sequences(current, gisaid, ncbi, genoflu_results):

    new_from_databases = []

    # Subset down to A6 only
    a6 = genoflu_results["Strain"].loc[genoflu_results["Genotype"] == "A6"].to_list()
    a6_gisaid = gisaid.loc[gisaid["Isolate_Name"].str.replace(" ", "_").isin(a6)]
    a6_ncbi = ncbi.loc[ncbi["GenBank_Title"].str.replace(" ", "_").str.split("(", expand = True)[1].isin(a6)]
    
    gisaid_strains = a6_gisaid["Isolate_Name"].to_list()
    ncbi_strains = set(a6_ncbi["GenBank_Title"].str.split("(", expand = True)[1].to_list())

    for i in gisaid_strains:
        if i in set(current["Isolate_Name"]):
            pass
        else:
            new_from_databases.append(i)

    print(ncbi_strains) # NCBI is a little difficult because the strain names were changed, but there are relatively few of them so I will add by hand
    
    print(f"Number of new sequences from GISAID: {len(new_from_databases)}")

    return(new_from_databases)

In [214]:
genoflu_results = pd.read_csv("../GenoFLU-multi/h5n5/results/results.tsv", sep = "\t")

In [215]:
new_sequences = identify_new_public_sequences(metadata, gisaid, ncbi, genoflu_results)

{'A/Common Eider/MA/25-021407-028-original/2025', 'A/Washington/2148/2025', 'A/Great Black-Backed Gull/MA/25-019390-018-original/2025', 'A/Common Raven/MA/25-019390-017-original/2025', 'A/Great black-backed gull/MA/25-021407-025-original/2025', 'A/Bufflehead/MA/25-007118-044-original/2025', 'A/Turkey/WA/25G04434-001-v/2025', 'A/Turkey Vulture/CT/25-006834-003-original/2025', 'A/Red-breasted Merganser/MA/25-021407-029-original/2025', 'A/bald_eagle/Ohio/OH25-5320/2025', 'A/Bald Eagle/OH/25-005188-001-original/2025', 'A/Northern Fulmar/Germany-NI/2024AI04273/2024', 'A/Common Raven/MA/25-019390-016-original/2025', 'A/Black Vulture/MA/25-021407-019-original/2025', 'A/Great black-backed gull/MA/25-021407-026-original/2025'}
Number of new sequences from GISAID: 13


In [216]:
new_sequences.extend(["A/Great black-backed gull/MA/25-021407-025-original/2025",
                      "A/Turkey/WA/25G04434-001-v/2025",
                      "A/Turkey Vulture/CT/25-006834-003-original/2025",
                      "A/Red-breasted Merganser/MA/25-021407-029-original/2025"
                     ])

In [217]:
print(new_sequences)

['A/black-headed_gull/Denmark/00376-1.01/2026', 'A/Common_Buzzard/England/133477/2024', 'A/Gannet/Scotland/124903/2023', 'A/Arctic_fox/Norway/2025-04-19222-1-2/2025', 'A/Arctic_fox/Norway/2025-04-21457-1-4/2025', 'A/large-billed crow/Hokkaido/B254/2026', 'A/carrion crow/Hokkaido/B257/2026', 'A/large-billed crow/Hokkaido/0103B118/2024', 'A/large-billed crow/Iwate/0303G001/2024', 'A/bald eagle/Ohio/OH25-5320/2025', 'A/large-billed crow/Iwate/0303G007/2024', 'A/large-billed crow/Iwate/0303G005/2024', 'A/Glaucous_gull/Norway/2025-07-2990-5-1/2025', 'A/Great black-backed gull/MA/25-021407-025-original/2025', 'A/Turkey/WA/25G04434-001-v/2025', 'A/Turkey Vulture/CT/25-006834-003-original/2025', 'A/Red-breasted Merganser/MA/25-021407-029-original/2025']


In [218]:
# Some of these have slightly updated location information on GenBank that was not in GISAID 
# And I think Alvin ended up pulling them from GISAID

update_metadata_ncbi = ["A/common eider/USA/021407-028/2025", 
                        "A/great black-backed gull/USA/019390-018/2025", 
                        "A/common raven/USA/019390-017/2025",
                        "A/Bufflehead/MA/25-007118-044-original/2025", 
                        "A/Bald Eagle/OH/25-005188-001-original/2025",
                        "A/Common Raven/MA/25-019390-016-original/2025", 
                        "A/Great black-backed gull/MA/25-021407-026-original/2025",
                        "A/Black Vulture/MA/25-021407-019-original/2025"
                       ]

In [219]:
# Alright
# Now I have a list of sequences that are not in the metadata at all and need to be added
# So I pull their FASTA sequences from the concatenated FASTA file 
# And I pull their metadata
# And I concatenate to current metadata

In [312]:
def add_new_public_sequences(new, current, gisaid, ncbi, concatenated_fastas):

    # First pull the metadata from gisaid and ncbi
    new_from_gisaid = gisaid.loc[gisaid["Isolate_Name"].str.replace(" ", "_").isin(new)] # Some have underscores originally
    new_from_gisaid = gisaid.loc[gisaid["Isolate_Name"].isin(new)] # some have spaces originally
    new_from_ncbi = ncbi.loc[ncbi["GenBank_Title"].str.split("(", expand = True)[1].isin(new)].drop_duplicates(subset = "SRA_Accession", keep = "first")

    new_from_all = pd.concat([new_from_gisaid, new_from_ncbi])

    # Then get the metadata you want from this
    columns = ["Isolate_Id", "Isolate_Name", "Collection_Date", "Subtype", "Genotype", "Location", "Host", 
              "GenBank_Title", "Accession", "Country", "Segment"]
    new_metadata = pd.concat([current, new_from_all[columns]])
    print(f"Metadata goes from {len(current)} to {len(new_metadata)} samples")
    
    # Pull sequences from FASTAs
    new_fastas = {}
    
    for f in concatenated_fastas.values():
        segment = f.loc[f["segment"].astype(str).str.contains(r"[A-Za-z]"), "segment"].iloc[0]
        new_fastas[segment] = {}
        seq_to_pull = f.loc[(f["identifier"].str.replace(">", "").isin(new)) | (f["identifier"].str.replace(">", "").str.replace("_", " ").isin(new))]
        new_fastas[segment] = seq_to_pull

    return(new_metadata, new_fastas)

In [313]:
new_metadata, new_fastas = add_new_public_sequences(new_sequences, metadata, gisaid, ncbi, concatenated_fastas)

Metadata goes from 342 to 359 samples


In [316]:
new_metadata.to_csv("./metadata/updated_metadata_uncleaned.tsv", sep = "\t")

In [315]:
for segment, df in new_fastas.items():
    label = str(segment)
    fasta_writer("./alignments/", f"{label}_fastas_to_add.fasta", df)

In [482]:
# Function made with help from Chat GPT

def add_new_tufts_sequences(strains, metadata):

    # Read in FASTAs and extract the Tufts code
    fasta_reader("./database_pulls/Nicole_2026-03-12/25ne00305COEIconsensus.fasta", "NE00305")
    fasta_reader("./database_pulls/Nicole_2026-03-12/25HP00818GBBGconsensus.fasta", "HP00818")
    fasta_reader("./database_pulls/Nicole_2026-03-12/24WP00294BLVUconsensus.fasta", "WP00294")
    fasta_reader("./database_pulls/Nicole_2026-03-12/24WI00326HERGconsensus.fasta", "WI00326")
    fasta_reader("./database_pulls/Nicole_2026-03-12/24WI00314Sanderlingconsensus.fasta", "WI00314")
    fasta_reader("./database_pulls/Nicole_2026-03-12/24WI00282GBBGconsensus.fasta", "WI00282")
    fasta_reader("./database_pulls/Nicole_2026-03-12/23NE01733GBBGconsensus.fasta", "NE01733")
    
    list_of_fastas = [NE00305, HP00818, WP00294, WI00326, WI00314, WI00282, NE01733]
    
    # Read in metadata and filter to just those in the list
    tufts_meta = pd.read_excel(metadata)
    tufts_meta_filtered = tufts_meta[tufts_meta["Tufts sample ID"].isin(strains)]
    
    # Merge on Tufts code
    merged_fastas = {}
    
    for fasta in list_of_fastas:
        # Extract a Tufts sample ID for the FASTA to match to Tufts metadata
        fasta["Tufts sample ID"] = fasta["header"].str.extract(r"(\d{2}\w{2}\d{5})")
        fasta["Tufts sample ID"] = fasta["Tufts sample ID"].str.upper()
        fasta["segment"] = fasta["header"].str.split("|", expand = True)[3]

        fasta_meta = fasta.merge(tufts_meta_filtered, on = "Tufts sample ID", how = "inner")

        # Generate an ID so you can match to your own metadata to add headers later on
        fasta_meta["id"] = ">" + fasta_meta["Tufts sample ID"].str.split("-", expand = True)[0]
        fasta_meta.loc[fasta_meta["NVSL sample ID"].str.contains(r"(\d{2}\-\d{6}\-\d{3})", regex = True), "id"] = ">" + fasta_meta["NVSL sample ID"]
        
        for code in fasta_meta["Tufts sample ID"].unique():

            sample_df = fasta_meta[fasta_meta["Tufts sample ID"] == code]

            if code not in merged_fastas:
                merged_fastas[code] = {}

            for segment in sample_df["segment"].unique():
                segment_fasta = sample_df[sample_df["segment"] == segment]

                merged_fastas[code][segment] = segment_fasta

    # Concatenate each segment across codes
    segments_combined = {}

    for code, seg_dict in merged_fastas.items():
        for segment, df in seg_dict.items():
            segments_combined.setdefault(segment, []).append(df)

    # concatenate
    segments_combined = {segment: pd.concat(dfs, ignore_index=True) for segment, dfs in segments_combined.items()}


    return(segments_combined)

In [483]:
fasta_list = []
strains = ["23NE01733","24WI00282","24WI00314","24WI00326","24WP00294","25HP00818","25NE00305"]
meta = "./database_pulls/Nicole_2026-03-12/h5n5_missingseqs-fromnicole_filledin.xlsx"

tufts_sequences_by_segment = add_new_tufts_sequences(strains, meta)

/var/folders/8z/j5vh6t990rj40lqqxtsr69z00000gp/T/ipykernel_3912/2015219901.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
/var/folders/8z/j5vh6t990rj40lqqxtsr69z00000gp/T/ipykernel_3912/2015219901.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
/var/folders/8z/j5vh6t990rj40lqqxtsr69z00000gp/T/ipykernel_3912/2015219901.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
/var/folders/8z/j5vh6t990rj40lqqxtsr69z00000gp/T/ipykernel_3912/2015219901.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
/var/folders/8z/j5vh6t990rj40lqqxtsr69z00000gp/T/ipykernel_3912/2015219901.py:33: UserWarning: This pattern is interpreted as a regular expr

In [484]:
for segment, seg_df in tufts_sequences_by_segment.items():
    try:  
        os.mkdir("./alignments/2026-03-16/")

    except OSError as error:
        pass

    with open(f"./alignments/2026-03-16/{segment}_from_tufts.fasta", "w") as f:
        for index, row in seg_df.iterrows():
            f.write(f"{row['id']}\n") # save with your own ID to match headers
            f.write(f"{row['sequence']}\n")

In [493]:
# Function: this code is for adding the correct headers

def correct_headers(METADATA):
    
    # Required: updated_metadata_uncleaned and fastas
    
    # fasta_reader("./alignments/2026-03-16/HA_fastas_to_add.fasta", "fastaHA")
    # fasta_reader("./alignments/2026-03-16/MP_fastas_to_add.fasta", "fastaMP")
    # fasta_reader("./alignments/2026-03-16/NA_fastas_to_add.fasta", "fastaNA")
    # fasta_reader("./alignments/2026-03-16/NP_fastas_to_add.fasta", "fastaNP")
    # fasta_reader("./alignments/2026-03-16/NS_fastas_to_add.fasta", "fastaNS")
    # fasta_reader("./alignments/2026-03-16/PA_fastas_to_add.fasta", "fastaPA")
    # fasta_reader("./alignments/2026-03-16/PB1_fastas_to_add.fasta", "fastaPB1")
    # fasta_reader("./alignments/2026-03-16/PB2_fastas_to_add.fasta",  "fastaPB2")

    fasta_reader("./alignments/2026-03-16/HA_from_tufts.fasta", "fastaHA")
    fasta_reader("./alignments/2026-03-16/MP_from_tufts.fasta", "fastaMP")
    fasta_reader("./alignments/2026-03-16/NA_from_tufts.fasta", "fastaNA")
    fasta_reader("./alignments/2026-03-16/NP_from_tufts.fasta", "fastaNP")
    fasta_reader("./alignments/2026-03-16/NS_from_tufts.fasta", "fastaNS")
    fasta_reader("./alignments/2026-03-16/PA_from_tufts.fasta", "fastaPA")
    fasta_reader("./alignments/2026-03-16/PB1_from_tufts.fasta", "fastaPB1")
    fasta_reader("./alignments/2026-03-16/PB2_from_tufts.fasta",  "fastaPB2")

    metadata = pd.read_csv(METADATA, sep = ",")

    dictionary = {"PB2": fastaPB2, "PB1": fastaPB1, "PA": fastaPA, "HA": fastaHA, "NP": fastaNP, "NA": fastaNA, "MP": fastaMP, "NS": fastaNS}

    # Merge on 
    merged_dict = {}
    
    for key, f in dictionary.items():
        # f["Isolate_Name"] = f["header"].astype(str).str.replace(">", "")
        # merged = f.merge(metadata, on = "Isolate_Name", how = "inner")
        f["Isolate_Id"] = f["header"].astype(str).str.replace(">", "")
        merged = f.merge(metadata, on = "Isolate_Id", how = "inner")
        
        merged["new-header"] = ">"+ merged["newid"]
        merged_dict[key] = merged        

    # Print new FASTA with newid as the header
    for segment, df in merged_dict.items():
        try:  
            os.mkdir("./alignments/2026-03-16/")
    
        except OSError as error:
            pass
    
        with open(f"./alignments/2026-03-16/new-headers_{segment}_from_tufts.fasta", "w") as f:
            for index, row in df.iterrows():
                f.write(f"{row['new-header']}\n")
                f.write(f"{row['sequence']}\n")

In [494]:
FILE = "./metadata/updated_metadata_cleaned.csv"

correct_headers(FILE)